# Una solución empresarial completa

## Ahora llevaremos nuestro proyecto del día 1 al siguiente nivel

### DESAFÍO EMPRESARIAL:

Crear un producto que genere un folleto para una empresa que se utilizará para posibles clientes, inversores y posibles reclutas.

Se nos proporcionará un nombre de empresa y su sitio web principal.

Consulte el final de este cuaderno para ver ejemplos de aplicaciones empresariales del mundo real.

Y recuerde: ¡siempre estoy disponible si tiene problemas o ideas! No dude en comunicarse conmigo.

In [1]:
# imports
# Si esto falla, verifica que esté ejecutándose desde un entorno "activado" con (llms) en el símbolo del sistema

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
# Inicialización y constantes and constants

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key[:8]=='sk-proj-':
    print("La clave de API parece buena")
else:
    print("¿Puede haber un problema con tu clave API? ¡Visita el cuaderno de resolución de problemas!")
    
MODEL = 'gpt-4o-mini'
openai = OpenAI()

La clave de API parece buena


In [3]:
# La clase para representar una Página Web

class Website:
    """
    Una clase de utilidad para representar un sitio web que hemos scrappeado, ahora con enlaces
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "Sin título"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Título de la Web:\n{self.title}\nContenido de la Web:\n{self.text}\n\n"

In [4]:
frog = Website("https://cursos.frogamesformacion.com")
print(frog.get_contents())
frog.links

Título de la Web:
Frogames
Contenido de la Web:
Ir al contenido principal
Frogames
Menú alternativo
Menú
Iniciar sesión
Ganadora del premio 'Enseñanza online de datos y competencias digitales más innovadora de Europa, 2023'
Pasión por
aprender
con los
mejores
En Frogames Formación te ayudamos a convertirte en todo un experto en: Programación de Videojuegos, Inteligencia Artificial, Machine Learning, Desarrollo de Apps, Data Science y mucho más.
Aprende mientras te diviertes
Cursos, Rutas y Suscripciones
Certificados de finalización
Qué encontrarás
dentro
de Frogames
Cursos online y formación de calidad para toda la família
Rutas temáticas
Rutas organizadas para que aprendas paso a paso, subiendo cada escalón e incrementando tus conocimientos adquiridos
Instructores Expertos
Con un equipo de profesionales y expertos en la materia que te acompañará a lo largo de todo el aprendizaje en la plataforma
Certificados blockchain
Títulos verificados por blockchain para cada habilidad que aprenda

['#main-content',
 '/',
 '/',
 '/users/sign_in',
 'https://cursos.frogamesformacion.com/pages/rutas',
 'https://cursos.frogamesformacion.com/pages/instructores',
 'https://cursos.frogamesformacion.com/pages/certificaciones',
 'https://cursos.frogamesformacion.com/collections',
 '/courses/prompt-engineering-android',
 '/courses/prompt-engineering-android',
 '/courses/ia-produccion',
 '/courses/ia-produccion',
 '/courses/power-bi-forecasting',
 '/courses/power-bi-forecasting',
 '/courses/analitica-avanzada-pbi',
 '/courses/analitica-avanzada-pbi',
 '/courses/power-bi-proyecto-fitness',
 '/courses/power-bi-proyecto-fitness',
 '/courses/ia-ejecutivos',
 '/courses/ia-ejecutivos',
 '/courses/geometria-analitica-desde-cero',
 '/courses/geometria-analitica-desde-cero',
 '/courses/agentes-ia',
 '/courses/agentes-ia',
 '/courses/domina-android-desde-cero-kotlin-compose',
 '/courses/domina-android-desde-cero-kotlin-compose',
 '/courses/unity-ui',
 '/courses/unity-ui',
 '/courses/matematicas-ml-3'

## Primer paso: hacer que GPT-4o-mini determine qué enlaces son relevantes

### Usar una llamada a gpt-4o-mini para leer los enlaces en una página web y responder en JSON estructurado.
Debería decidir qué enlaces son relevantes y reemplazar los enlaces relativos como "/about" con "https://company.com/about".
Usaremos "one shot prompting" en las que proporcionamos un ejemplo de cómo debería responder en la solicitud.

Este es un excelente caso de uso para un LLM, porque requiere una comprensión matizada. Imagínate intentar programar esto sin LLMs analizando la página web: ¡sería muy difícil!

Nota al margen: existe una técnica más avanzada llamada "Salidas estructuradas" en la que requerimos que el modelo responda de acuerdo con una especificación. Cubrimos esta técnica en la Semana 8 durante nuestro proyecto autónomo de inteligencia artificial Agentic.

In [5]:
link_system_prompt = "Se te proporciona una lista de enlaces que se encuentran en una página web. \
Puedes decidir cuáles de los enlaces serían los más relevantes para incluir en un folleto sobre la empresa, \
como enlaces a una página Acerca de, una página de la empresa, las carreras/empleos disponibles o páginas de Cursos/Packs.\n"
link_system_prompt += "Debes responder en JSON como en este ejemplo:"
link_system_prompt += """
{
    "links": [
        {"type": "Pagina Sobre nosotros", "url": "https://url.completa/aqui/va/sobre/nosotros"},
        {"type": "Pagina de Cursos": "url": "https://otra.url.completa/courses"}
    ]
}
"""

In [6]:
print(link_system_prompt)

Se te proporciona una lista de enlaces que se encuentran en una página web. Puedes decidir cuáles de los enlaces serían los más relevantes para incluir en un folleto sobre la empresa, como enlaces a una página Acerca de, una página de la empresa, las carreras/empleos disponibles o páginas de Cursos/Packs.
Debes responder en JSON como en este ejemplo:
{
    "links": [
        {"type": "Pagina Sobre nosotros", "url": "https://url.completa/aqui/va/sobre/nosotros"},
        {"type": "Pagina de Cursos": "url": "https://otra.url.completa/courses"}
    ]
}



In [7]:
def get_links_user_prompt(website):
    user_prompt = f"Aquí hay una lista de enlaces de la página web {website.url} - "
    user_prompt += "Por favor, decide cuáles de estos son enlaces web relevantes para un folleto sobre la empresa. Responde con la URL https completa en formato JSON. \
No incluyas Términos y Condiciones, Privacidad ni enlaces de correo electrónico.\n"
    user_prompt += "Links (puede que algunos sean links relativos):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [8]:
print(get_links_user_prompt(frog))

Aquí hay una lista de enlaces de la página web https://cursos.frogamesformacion.com - Por favor, decide cuáles de estos son enlaces web relevantes para un folleto sobre la empresa. Responde con la URL https completa en formato JSON. No incluyas Términos y Condiciones, Privacidad ni enlaces de correo electrónico.
Links (puede que algunos sean links relativos):
#main-content
/
/
/users/sign_in
https://cursos.frogamesformacion.com/pages/rutas
https://cursos.frogamesformacion.com/pages/instructores
https://cursos.frogamesformacion.com/pages/certificaciones
https://cursos.frogamesformacion.com/collections
/courses/prompt-engineering-android
/courses/prompt-engineering-android
/courses/ia-produccion
/courses/ia-produccion
/courses/power-bi-forecasting
/courses/power-bi-forecasting
/courses/analitica-avanzada-pbi
/courses/analitica-avanzada-pbi
/courses/power-bi-proyecto-fitness
/courses/power-bi-proyecto-fitness
/courses/ia-ejecutivos
/courses/ia-ejecutivos
/courses/geometria-analitica-desde

In [9]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [10]:
anthropic = Website("https://anthropic.com")
anthropic.links

['#main',
 '#footer',
 'https://www.anthropic.com/',
 'https://www.anthropic.com/research',
 'https://www.anthropic.com/economic-futures',
 'https://www.anthropic.com/constitution',
 'https://www.anthropic.com/transparency',
 'https://www.anthropic.com/responsible-scaling-policy',
 'http://trust.anthropic.com/',
 'https://www.anthropic.com/learn',
 'https://claude.com/resources/tutorials',
 'https://claude.com/resources/use-cases',
 'https://www.anthropic.com/engineering',
 'https://platform.claude.com/docs',
 'https://www.anthropic.com/company',
 'https://www.anthropic.com/careers',
 'https://www.anthropic.com/events',
 'https://www.anthropic.com/news',
 'https://claude.ai',
 'https://claude.com/product/overview',
 'https://claude.com/product/claude-code',
 'https://claude.com/product/cowork',
 'https://claude.com/product/claude-security',
 'https://claude.com/platform/api',
 'https://claude.com/pricing',
 'https://claude.com/contact-sales',
 'https://www.anthropic.com/claude/opus',
 

In [11]:
get_links("https://anthropic.com")

{'links': [{'type': 'Página de la empresa',
   'url': 'https://www.anthropic.com/company'},
  {'type': 'Carreras', 'url': 'https://www.anthropic.com/careers'},
  {'type': 'Página sobre nosotros', 'url': 'https://www.anthropic.com/learn'},
  {'type': 'Página de Investigación',
   'url': 'https://www.anthropic.com/research'},
  {'type': 'Eventos', 'url': 'https://www.anthropic.com/events'},
  {'type': 'Noticias', 'url': 'https://www.anthropic.com/news'}]}

In [12]:
get_links("https://cursos.frogamesformacion.com")

{'links': [{'type': 'Pagina Principal',
   'url': 'https://cursos.frogamesformacion.com'},
  {'type': 'Pagina de Instructores',
   'url': 'https://cursos.frogamesformacion.com/pages/instructores'},
  {'type': 'Pagina de Certificaciones',
   'url': 'https://cursos.frogamesformacion.com/pages/certificaciones'},
  {'type': 'Clientes',
   'url': 'https://cursos.frogamesformacion.com/pages/nuestros-clientes'},
  {'type': 'Frogames para Empresas',
   'url': 'https://cursos.frogamesformacion.com/pages/frogames-para-empresas'},
  {'type': 'Premios',
   'url': 'https://cursos.frogamesformacion.com/pages/premios'},
  {'type': 'Convalidación de Cursos de Udemy',
   'url': 'https://cursos.frogamesformacion.com/courses/convalidacion-de-cursos-de-udemy'},
  {'type': 'Pagina de Cursos',
   'url': 'https://cursos.frogamesformacion.com/courses/prompt-engineering-android'},
  {'type': 'Pagina de Cursos',
   'url': 'https://cursos.frogamesformacion.com/collections'},
  {'type': 'Rutas de Aprendizaje',
  

## Segundo paso: ¡crea el folleto!

Reúne todos los detalles en otro mensaje para GPT4-o

In [13]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Links encontrados:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [14]:
print(get_all_details("https://anthropic.com"))

Links encontrados: {'links': [{'type': 'Pagina de Inicio', 'url': 'https://www.anthropic.com/'}, {'type': 'Pagina de Investigación', 'url': 'https://www.anthropic.com/research'}, {'type': 'Pagina de la Empresa', 'url': 'https://www.anthropic.com/company'}, {'type': 'Carreras', 'url': 'https://www.anthropic.com/careers'}, {'type': 'Pagina de Eventos', 'url': 'https://www.anthropic.com/events'}, {'type': 'Pagina de Noticias', 'url': 'https://www.anthropic.com/news'}, {'type': 'Pagina de Aprendizaje', 'url': 'https://www.anthropic.com/learn'}, {'type': 'Pagina de Transparencia', 'url': 'https://www.anthropic.com/transparency'}, {'type': 'Pagina de Futuras Económicas', 'url': 'https://www.anthropic.com/economic-futures'}, {'type': 'Pagina de Escala Responsable', 'url': 'https://www.anthropic.com/responsible-scaling-policy'}]}
Landing page:
Título de la Web:
Home \ Anthropic
Contenido de la Web:
Skip to main content
Skip to footer
Research
Economic Futures
Commitments
Initiatives
Claude's C

In [15]:
system_prompt = "Eres un asistente que analiza el contenido de varias páginas relevantes del sitio web de una empresa\
y crea un folleto breve sobre la empresa para posibles clientes, inversores y nuevos empleados. Responde en formato Markdown.\
Incluye detalles sobre la cultura de la empresa, los clientes, las carreras/empleos y los cursos/packs para futuros empleos si tienes la información."

# O descomenta las líneas a continuación para obtener un folleto más humorístico: esto demuestra lo fácil que es incorporar el "tono":

# system_prompt = "Eres un asistente que analiza el contenido de varias páginas relevantes del sitio web de una empresa \
# y crea un folleto breve, divertido y gracioso sobre la empresa para posibles clientes, inversores y nuevos empleados. Responde en formato Markdown.\
#Incluye detalles sobre la cultura de la empresa, los clientes y los cursos/packs para futuros empleos si tienes la información."


In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"Estás mirando una empresa llamada: {company_name}\n"
    user_prompt += f"Aquí se encuentra el contenido de su página de inicio y otras páginas relevantes; usa esta información para crear un breve folleto de la empresa en Markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:20_000] # Truncar si tiene más de 20.000 caracteres
    return user_prompt

In [17]:
get_brochure_user_prompt("Anthropic", "https://anthropic.com")

Links encontrados: {'links': [{'type': 'Página de la empresa', 'url': 'https://www.anthropic.com/company'}, {'type': 'Página de Carreras', 'url': 'https://www.anthropic.com/careers'}, {'type': 'Página de Investigación', 'url': 'https://www.anthropic.com/research'}, {'type': 'Página de Eventos', 'url': 'https://www.anthropic.com/events'}, {'type': 'Página de Aprendizaje', 'url': 'https://www.anthropic.com/learn'}, {'type': 'Página de noticias', 'url': 'https://www.anthropic.com/news'}, {'type': 'Página de Futuras Económicas', 'url': 'https://www.anthropic.com/economic-futures'}, {'type': 'Página de Transparencia', 'url': 'https://www.anthropic.com/transparency'}, {'type': 'Página de Política de Escalado Responsable', 'url': 'https://www.anthropic.com/responsible-scaling-policy'}]}


"Estás mirando una empresa llamada: Anthropic\nAquí se encuentra el contenido de su página de inicio y otras páginas relevantes; usa esta información para crear un breve folleto de la empresa en Markdown.\nLanding page:\nTítulo de la Web:\nHome \\ Anthropic\nContenido de la Web:\nSkip to main content\nSkip to footer\nResearch\nEconomic Futures\nCommitments\nInitiatives\nClaude's Constitution\nTransparency\nResponsible Scaling Policy\nTrust center\nSecurity and compliance\nLearn\nLearn\nAnthropic Academy\nTutorials\nUse cases\nEngineering at Anthropic\nDeveloper docs\nCompany\nAbout\nCareers\nEvents\nNews\nTry Claude\nTry Claude\nTry Claude\nLearn more about Claude\nProducts\nClaude\nClaude Code\nClaude Cowork\nClaude Security\nClaude Platform\nPricing\nContact sales\nModels\nOpus\nSonnet\nHaiku\nLog in\nClaude.ai\nClaude Console\nEN\nThis is some text inside of a div block.\nLog in to Claude\nLog in to Claude\nLog in to Claude\nDownload app\nDownload app\nDownload app\nResearch\nEconom

In [18]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [19]:
create_brochure("Anthropic", "https://anthropic.com")

Links encontrados: {'links': [{'type': 'Página de la empresa', 'url': 'https://www.anthropic.com/company'}, {'type': 'Página de Carreras', 'url': 'https://www.anthropic.com/careers'}, {'type': 'Página de Investigación', 'url': 'https://www.anthropic.com/research'}, {'type': 'Página de Eventos', 'url': 'https://www.anthropic.com/events'}, {'type': 'Página de Aprendizaje', 'url': 'https://www.anthropic.com/learn'}]}


# Bienvenido a Anthropic

## Sobre Nosotros
Anthropic es una empresa de investigación y seguridad en inteligencia artificial (IA) que crea sistemas de IA confiables, interpretables y controlables. Nuestro compromiso es asegurar que los beneficios de la IA se materialicen, mientras mitigamos sus riesgos potenciales. Como una corporación de beneficio público, nuestra misión es contribuir al bienestar a largo plazo de la humanidad a través del desarrollo responsable de IA.

## Cultura Corporativa
En Anthropic, trabajamos bajo principios que guían nuestras decisiones y acciones:
1. **Actuar por el Bien Global:** Tomamos decisiones que maximicen los resultados positivos para la humanidad en el largo plazo.
2. **Mantener el Equilibrio:** Reconocemos los riesgos y beneficios que la IA puede traer, buscando un enfoque equilibrado.
3. **Ser amables con nuestros usuarios:** Cultivamos la generosidad en todas nuestras interacciones, reconociendo que nuestros "usuarios" abarcan desde clientes hasta comunidades afectadas por nuestra tecnología.
4. **Inspirar una Competencia por la Seguridad:** Buscamos establecer un estándar elevado en la seguridad de los sistemas de IA y motivar a otros a seguir nuestro ejemplo.
5. **Simplicidad que Funciona:** Abordamos problemas de forma empírica, priorizando el impacto sobre la sofisticación de los métodos.
6. **Ser útiles, honestos y inofensivos:** Fomentamos un ambiente de trabajo de alta confianza y baja ego, donde todos contribuyen activamente.
7. **Colocar la Misión en Primer Lugar:** La misión es nuestra razón de ser, dándonos un propósito compartido para actuar rápidamente y con colaboración.

## Clientes
Nuestros productos, que incluyen Claude, Claude Code, y Claude Security, están diseñados para servir a una variedad de sectores, incluidos negocios, organizaciones sin fines de lucro y agencias gubernamentales. Nuestra visión es ayudar a estos grupos a manejar de manera efectiva y segura las nuevas tecnologías de IA.

## Oportunidades de Carrera
En Anthropic, buscamos individuos apasionados por resolver problemas difíciles con un verdadero impacto. Ofrecemos un entorno de trabajo inclusivo y colaborativo, donde las ideas de todos son valoradas. Nuestros beneficios incluyen:
- **Salud y bienestar:** Seguro médico integral y beneficios de fertilidad inclusivos.
- **Compensación competitiva:** Salarios competitivos y paquetes de equidad.
- **Apoyo adicional:** Estipendios para educación y oficina en casa, así como apoyo para reubicación.

La diversidad en backgrounds es fomentada, y valoramos a aquellos con experiencia en investigación, ingeniería o roles no técnicos que demuestren una interés genuino en nuestra misión.

## Formación y Cursos
Anthropic Academy ofrece tutoriales y recursos educativos para ayudar a aquellos que desean familiarizarse con nuestras tecnologías y contribuir a un futuro seguro con IA. Fomentamos el aprendizaje continuo, proporcionando a nuestros empleados las herramientas y conocimientos necesarios para prosperar en sus carreras.

## Conclusión
Si estás interesado en formar parte de un equipo que está dando forma al futuro de la inteligencia artificial con un enfoque en la seguridad y el bien común, ¡nos encantaría conocerte!

[Explora las Oportunidades Abiertas](https://www.anthropic.com/careers)  |  [Conoce más sobre Claude](https://www.anthropic.com/products)  |  [Visita Anthropic Academy](https://www.anthropic.com/learn)

In [20]:
create_brochure("Frogames Formación", "https://cursos.frogamesformacion.com")

Links encontrados: {'links': [{'type': 'Página Principal', 'url': 'https://cursos.frogamesformacion.com'}, {'type': 'Página de Rutas', 'url': 'https://cursos.frogamesformacion.com/pages/rutas'}, {'type': 'Página de Instructores', 'url': 'https://cursos.frogamesformacion.com/pages/instructores'}, {'type': 'Página de Certificaciones', 'url': 'https://cursos.frogamesformacion.com/pages/certificaciones'}, {'type': 'Página de Nuestros Clientes', 'url': 'https://cursos.frogamesformacion.com/pages/nuestros-clientes'}, {'type': 'Página Frogames para Empresas', 'url': 'https://cursos.frogamesformacion.com/pages/frogames-para-empresas'}, {'type': 'Página de Premios', 'url': 'https://cursos.frogamesformacion.com/pages/premios'}, {'type': 'Página de Afiliados', 'url': 'https://cursos.frogamesformacion.com/pages/afiliados'}, {'type': 'Curso de Convalidación de Cursos de Udemy', 'url': 'https://cursos.frogamesformacion.com/courses/convalidacion-de-cursos-de-udemy'}, {'type': 'Cursos Disponibles', 'u

# Frogames Formación

## ¡Bienvenido a Frogames Formación!

### Premios y Reconocimientos
Frogames ha sido galardonada con el premio **'Enseñanza online de datos y competencias digitales más innovadora de Europa, 2023'**. Este reconocimiento refleja la dedicación de Frogames a proporcionar educación de calidad y vanguardista.

### Nuestra Misión
En **Frogames**, nos apasiona ayudar a nuestros estudiantes a convertirse en expertos en áreas altamente demandadas como:
- Programación de Videojuegos
- Inteligencia Artificial
- Machine Learning
- Desarrollo de Aplicaciones
- Data Science y mucho más.

### ¿Qué Ofrecemos?
- **Cursos Online**: Formación de calidad para toda la familia.
- **Rutas de Aprendizaje**: Programas organizados para guiar a los estudiantes a través de su proceso de aprendizaje.
- **Instructores Expertos**: Un equipo de profesionales que acompaña a los estudiantes en su trayectoria.
- **Certificados Blockchain**: Diplomas verificados listos para compartir en redes sociales y mejorar el CV.
- **Actualizaciones Constantes**: Nuevos cursos y actualizaciones regularmente.

### Testimonios de Estudiantes
Más de **500.000 estudiantes** satisfechos han compartido sus experiencias, como:
> “Me gusta aprender en Frogames porque sus cursos son especializados en contenido práctico y avanzado.” - **Edwyn Mendoza**
>
> “¡Me encanta aprender aquí! Con profesores que dominan el tema y ayudan en todo momento.” - **Eulogio Enamorado Pallares**

### Oportunidades Profesionales
#### Trabaja con Nosotros
Estamos en búsqueda de nuevos talentos para unirse a nuestro equipo. Además, puedes convertirte en **afiliado** y ser remunerado por cada venta que consigas. 

#### Rutas de Aprendizaje
Ofrecemos **17 rutas de aprendizaje** diseñadas para llevarte de cero a experto en tecnologías demandadas como:
- Análisis de Datos
- Desarrollo de Videojuegos
- Inteligencia Artificial
- Trading Algorítmico y más.

### Planes de Suscripción
- **Rana de Oro**: 224 productos didácticos por **€349/año**.
- **Rana de Plata**: 225 productos didácticos por **€199/6 meses**.
- **Rana de Bronce**: Acceso mensual a 224 productos por **€39/mes**.

### Comienza a Aprender
Te ofrecemos un curso gratuito para iniciar tu camino en el mundo del **trading algorítmico con Python**. ¡Descubre todo lo que Frogames puede ofrecerte!

Para más información, visita nuestro sitio web: [Frogames Formación](https://www.frogames.com) 

### Conclusión
¡Únete a la comunidad de Frogames y transforma tu futuro profesional a través de la educación online divertida y de calidad! ¡Esperamos verte pronto!

## Por último, una pequeña mejora

Con un pequeño ajuste, podemos cambiar esto para que los resultados se transmitan desde OpenAI,
con la animación de máquina de escribir habitual


In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("Anthropic", "https://anthropic.com")

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
stream_brochure("Frogames Formación", "https://cursos.frogamesformacion.com")

## Aplicaciones empresariales

En este ejercicio, ampliamos el código del día 1 para realizar múltiples llamadas a LLM y generar un documento.

En términos de técnicas, este es quizás el primer ejemplo de patrones de diseño de Agentic AI, ya que combinamos múltiples llamadas a LLM. Esto se abordará más en la semana 2 y luego volveremos a Agentic AI de manera importante en la semana 8, cuando construyamos una solución Agent completamente autónoma.

En términos de aplicaciones empresariales, generar contenido de esta manera es uno de los casos de uso más comunes. Al igual que con el resumen, esto se puede aplicar a cualquier vertical empresarial. Escriba contenido de marketing, genere un tutorial de producto a partir de una especificación, cree contenido de correo electrónico personalizado y mucho más. Explore cómo puede aplicar la generación de contenido a su negocio e intente crear un prototipo de prueba de concepto.